In [ ]:
from dotenv import load_dotenv
import os
import requests
from datasets import load_dataset
from math_arena_datasets import categories_2025
import pandas as pd
from sklearn.model_selection import train_test_split
from time import sleep
from openai import OpenAI

# client = OpenAI()

# --- Load API key ---
load_dotenv()
API_URL = "https://api.groq.com/openai/v1/chat/completions"
API_KEY = os.environ.get("GROQ_API_KEY")

# --- Build the few-shot prompt ---
def build_prompt(examples, query):
    prompt = (
        "You are a classifier that categorizes math problems into one of these: "
        "Arithmetic, Algebra, Geometry, Combinatorics, Number Theory, Probability, etc.\n\n"
    )
    for _, row in examples.iterrows():
        prompt += f"Problem: {row['problem']}\nCategory: {row['problem_type']}\n\n"
    prompt += f"Problem: {query}\nCategory:"
    return prompt

# --- Query the Groq API ---
def classify_with_groq(prompt):
    response = requests.post(
        API_URL,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": "llama-3.3-70b-versatile",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2,
        }
    )

    try:
        return response.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print("Error:", e, "Response:", response.text)
        return ""

# --- Load and merge all datasets ---
dfs = []
for path in categories_2025:
    dataset_dict = load_dataset(path)
    split = list(dataset_dict.keys())[0]
    df_split = dataset_dict[split].to_pandas()
    dfs.append(df_split)

df = pd.concat(dfs, ignore_index=True)
df["problem_type"] = df["problem_type"].apply(lambda x: x[0] if isinstance(x, list) else x)

# --- Keep only needed columns ---
df = df[["problem", "problem_type"]].dropna()

# --- Train/test split ---
train_df, test_df = pd.read_csv('data/problem_type_data.train.csv'), pd.read_csv('data/problem_type_data.test.csv')

# --- Evaluate with few-shot ICL ---
correct = 0
for _, row in test_df.iterrows():
    few_shots = train_df.sample(5)
    prompt = build_prompt(few_shots, row["problem"])
    predicted = classify_with_groq(prompt)
    print(f"Predicted: {predicted} | Actual: {row['problem_type']}")
    if predicted.lower() == row["problem_type"].lower():
        correct += 1
    sleep(5)

accuracy = correct / len(test_df)
print(f"Accuracy: {accuracy:.2%}")


d:\conda\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable